# EDGAR-CORPUS Parquet Downloader and Dataset Builder
Downloads the parquet conversion of eloukas/edgar-corpus, extracts 2010-2014 Item 1A risk factors, and saves a local research dataset.


In [ ]:
!pip install -q huggingface_hub pandas pyarrow tqdm nltk

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import re
import nltk
from nltk.tokenize import sent_tokenize
from huggingface_hub import snapshot_download
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

## Download parquet repository

In [ ]:
local_dir = snapshot_download(
    repo_id='eloukas/edgar-corpus',
    repo_type='dataset',
    revision='refs/convert/parquet'
)
print(local_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 150 files:   0%|          | 0/150 [00:00<?, ?it/s]

In [ ]:
parquet_files = list(Path(local_dir).rglob('*.parquet'))
print('Parquet files:', len(parquet_files))
for f in parquet_files[:20]:
    print(f)

## Inspect schema

In [ ]:
sample = pd.read_parquet(parquet_files[0])
print(sample.columns.tolist())
sample.head()

## Load required columns only

In [ ]:
dfs=[]
for file in tqdm(parquet_files):
    try:
        df=pd.read_parquet(file, columns=['cik','year','filename','section_1A'])
        dfs.append(df)
    except Exception as e:
        print(file, e)

edgar=pd.concat(dfs, ignore_index=True)
print(edgar.shape)

In [ ]:
print("Beginning year:", edgar["year"].min())
print("Ending year:", edgar["year"].max())

In [ ]:
edgar["year"] = pd.to_numeric(
    edgar["year"],
    errors="coerce"
)

edgar=edgar[edgar['year'].between(2010,2020)].copy()
edgar.rename(columns={'section_1A':'risk_text'}, inplace=True)
edgar=edgar[edgar['risk_text'].notna()].copy()
edgar['cik']=edgar['cik'].astype(str).str.zfill(10)

In [ ]:
def clean_text(text):
    text=re.sub(r'\s+',' ',str(text))
    return text.strip()

edgar['risk_text']=edgar['risk_text'].apply(clean_text)

In [ ]:
edgar['risk_word_count']=edgar['risk_text'].str.split().str.len()
edgar['risk_sentence_count']=edgar['risk_text'].apply(lambda x: len(sent_tokenize(str(x))))

In [ ]:
edgar=edgar[['cik','year','filename','risk_word_count','risk_sentence_count','risk_text']]
print(edgar.shape)
edgar.head()

## Save local research dataset

In [ ]:
edgar.to_parquet('edgar_risk_factors_2010_2020.parquet', index=False)
edgar.to_csv('edgar_risk_factors_2010_2020.csv', index=False)
print('Saved dataset files successfully')